In [2]:
import os
import certifi
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
import langchainhub as hub
from langchain.tools import tool
import requests

In [5]:
from langgraph.prebuilt import create_react_agent

# ======================================================
# LOAD ENV VARIABLES
# ======================================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [6]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3080\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [7]:
@tool 

def get_weather_data(city: str) -> str:
    """Fetch current weather information for a specified city."""
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    
    response = requests.get(url)
    data = response.json()
    
    if "current" not in data:
        return f"Không thể lấy dữ liệu thời tiết cho thành phố {city}"
    
    return (
        f"Thành phố: {city}\n"
        f"Nhiệt độ: {data['current']['temperature']}°C\n"
        f"Tình trạng thời tiết: {data['current']['weather_descriptions'][0]}\n"
        f"Độ ẩm: {data['current']['humidity']}%"
    )

In [8]:
# Khởi tạo mô hình Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [10]:
from langchainhub import Client

hub = Client()

In [11]:
# ======================================================
# PROMPT
# ======================================================

prompt = hub.pull("hwchase17/react")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3080\3539103103.py:5: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = hub.pull("hwchase17/react")
c:\Users\ADMIN\anaconda3\envs\langagent\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


In [12]:
tools = [search_tool,get_weather_data]

In [15]:
custom_prompt = (
    "You are a helpful assistant. "
    "Always use the available tools to answer questions about weather or recent events. "
    "Respond politely in Vietnamese."
)

In [17]:
# ======================================================
# CREATE AGENT
# ======================================================

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=custom_prompt
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3080\3423635905.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [19]:
response = agent.invoke({
    "messages": [("user", "thời tiết hà tĩnh hôm nay")]
})


c:\Users\ADMIN\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\ADMIN\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [ ]:
print(response["messages"][-1].content)